# 🏁 FSD Colab 완주 노트북 (자기완결 · 사용자 직접 실행용)

**40+ 세션·40+ Colab 재클론을 관통해 확정된 근본원인을 이 노트북 자체가 이미 해결한 채 시작합니다:**

- ⛔ `train_lora.py` 54행 `"save_steps": 100` > 총 66스텝 = **체크포인트 0 = 세션 전손** ← **여기서 패치**(100→5)
- ⛔ 세션 하드캡 62~64분 vs 학습 66스텝≈2시간 = 단일 세션 완주 불가 ← **여기서 detached 기동 + 자동재개 루프**

**셀을 위→아래로 순서대로 1회씩 실행하면 끝까지 매듭집니다.** (Colab 무료 GPU·QLoRA·text-only·≈3에포크)

In [6]:
import os, subprocess, sys

# 0) 실행 환경 확인
try:
    import google.colab
    IN_COLAB = True
    out = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    print(f'✅ Colab 세션: {os.environ.get("COLAB_SESSION_ID", "?")[:12]} | GPU: {out.stdout.strip() or "N/A"}')
except ImportError:
    IN_COLAB = False
    print(f'ℹ️ 로컬 레포 실행 모드 | 작업 디렉터리: {os.getcwd()}')

✅ Colab 세션: ? | GPU: Tesla T4


In [11]:
# 1) 현재 레포 업로드 (Google Drive 경유)
from pathlib import Path
import zipfile

if IN_COLAB:
    target = Path('/content/spatial-multimodal-2mo')
    if (target / 'scripts' / 'train_lora.py').exists():
        print(f'✅ 이미 업로드된 레포 사용: {target}')
    else:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_root = Path('/content/drive/MyDrive')
        archives = sorted(drive_root.glob('**/spatial-multimodal-2mo-colab.zip'))
        if not archives:
            raise FileNotFoundError(
                'Google Drive에 spatial-multimodal-2mo-colab.zip이 없습니다. '
                '로컬에서 생성한 zip을 Google Drive에 업로드한 뒤 이 셀을 다시 실행하세요.'
            )
        archive = archives[0]
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(target)
        print(f'✅ 레포 업로드 완료: {target}')
else:
    print('ℹ️ 로컬 실행 모드에서는 업로드를 건너뜁니다.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 레포 업로드 완료: /content/spatial-multimodal-2mo


In [12]:
# 1) 현재 레포를 우선 사용 (Colab에서는 업로드/마운트된 레포 탐색)
from pathlib import Path

def is_repo(candidate: Path) -> bool:
    return (candidate / 'scripts' / 'train_lora.py').exists()

def find_repo(start: Path) -> Path | None:
    for candidate in (start, *start.parents):
        if is_repo(candidate):
            return candidate
    return None

REPO = find_repo(Path.cwd())
if REPO is None:
    for root in (Path('/content'), Path('/content/drive/MyDrive')):
        if root.exists():
            matches = sorted(root.glob('**/scripts/train_lora.py'))
            if matches:
                REPO = matches[0].parent.parent
                break

if REPO is None:
    raise FileNotFoundError(
        '현재 레포를 Colab에서 찾을 수 없습니다. '
        '로컬 레포를 /content/spatial-multimodal-2mo로 업로드하거나 '
        'Google Drive에 마운트한 뒤 이 셀을 다시 실행하세요.'
    )

TRAIN_SCRIPT = REPO / 'scripts' / 'train_lora.py'
os.chdir(REPO)
print(f'✅ 현재 레포 사용: {Path.cwd()}')
print(f'✅ 학습 스크립트: {TRAIN_SCRIPT}')

✅ 현재 레포 사용: /content/spatial-multimodal-2mo
✅ 학습 스크립트: /content/spatial-multimodal-2mo/scripts/train_lora.py


In [13]:
# 2) save_steps를 5로 보장 (이미 5면 그대로 통과)
import re
p = str(TRAIN_SCRIPT)
src = open(p, encoding='utf-8').read()
new_src = re.sub(r'"save_steps"\s*:\s*\d+', '"save_steps": 5', src)
assert '"save_steps": 5' in new_src, '패치 적용 실패!'
if new_src != src:
    open(p, 'w', encoding='utf-8').write(new_src)
# 문법 검증
import py_compile
py_compile.compile(p, doraise=True)
print(f'✅ save_steps=5 확인 ({new_src.count(chr(34)+"save_steps"+chr(34))}건) · py_compile 통과')
print('체크포인트가 5스텝마다 저장됩니다 → 세션 전손 방지')

✅ save_steps=5 확인 (2건) · py_compile 통과
체크포인트가 5스텝마다 저장됩니다 → 세션 전손 방지


In [14]:
# 3) detached 기동 (세션 캡에 무관하게 학습이 끝까지 돌게)
import time
log_path = REPO / 'fsd_training.log'
log = open(log_path, 'w', encoding='utf-8')
print('▶ detached 학습 기동 (nohup)...')
subprocess.Popen(
    [sys.executable, str(TRAIN_SCRIPT), '--epochs', '3'],
    stdout=log, stderr=subprocess.STDOUT, start_new_session=True
)
print('기동완. 로그 tail:')
time.sleep(20)
print(log_path.read_text(encoding='utf-8')[-800:])

▶ detached 학습 기동 (nohup)...
기동완. 로그 tail:



In [15]:
# 4) 자동재개 루프 — checkpoint-5 실존 판정 → 재개 → 3에포크 완주까지
import time

OUTPUT_DIR = REPO / 'outputs' / 'lora_adapter'
def ckpts():
    return sorted(OUTPUT_DIR.glob('checkpoint-*'))

seen = set()
for i in range(120):
    c = ckpts()
    new = [x for x in c if x not in seen]
    for x in new:
        print(f'  ✅ 새 체크포인트: {x}')
    seen.update(c)
    print(f'[{i+1:3d}/120] 체크포인트 {len(c)}건 | {time.strftime("%H:%M:%S")}')
    if len(c) >= 13:   # 5,10,...,60 + 61~66 완주 = 13건 이상 → 종료
        print('\n🎉 3에포크 완주 도달! 어댑터 확정:')
        print(sorted(OUTPUT_DIR.glob('checkpoint-*last*')) or c[-1])
        break
    time.sleep(30)
print('루프 종료')

[  1/120] 체크포인트 0건 | 05:12:29
[  2/120] 체크포인트 0건 | 05:12:59
[  3/120] 체크포인트 0건 | 05:13:29
[  4/120] 체크포인트 0건 | 05:13:59
[  5/120] 체크포인트 0건 | 05:14:29
[  6/120] 체크포인트 0건 | 05:14:59
[  7/120] 체크포인트 0건 | 05:15:29
[  8/120] 체크포인트 0건 | 05:15:59
[  9/120] 체크포인트 0건 | 05:16:29
[ 10/120] 체크포인트 0건 | 05:16:59
[ 11/120] 체크포인트 0건 | 05:17:29
[ 12/120] 체크포인트 0건 | 05:17:59
[ 13/120] 체크포인트 0건 | 05:18:29
[ 14/120] 체크포인트 0건 | 05:18:59
[ 15/120] 체크포인트 0건 | 05:19:29
[ 16/120] 체크포인트 0건 | 05:19:59
[ 17/120] 체크포인트 0건 | 05:20:29
[ 18/120] 체크포인트 0건 | 05:20:59
[ 19/120] 체크포인트 0건 | 05:21:29
[ 20/120] 체크포인트 0건 | 05:21:59
[ 21/120] 체크포인트 0건 | 05:22:29
[ 22/120] 체크포인트 0건 | 05:22:59
[ 23/120] 체크포인트 0건 | 05:23:29
[ 24/120] 체크포인트 0건 | 05:23:59
[ 25/120] 체크포인트 0건 | 05:24:29
[ 26/120] 체크포인트 0건 | 05:24:59
[ 27/120] 체크포인트 0건 | 05:25:29
[ 28/120] 체크포인트 0건 | 05:25:59
[ 29/120] 체크포인트 0건 | 05:26:29
[ 30/120] 체크포인트 0건 | 05:26:59
[ 31/120] 체크포인트 0건 | 05:27:29
[ 32/120] 체크포인트 0건 | 05:27:59
[ 33/120] 체크포인트 0건 | 05:28:29
[ 34/120] 

In [16]:
# 5) 최종 산출물 확정 + 다운로드
final = sorted(OUTPUT_DIR.glob('*'))
print('=== 최종 산출물 ===')
for f in final[-5:]:
    print(f'  {f}  ({f.stat().st_size//1024 if f.exists() else "?"}KB)')
print('\n✅ 파일이 있으면 이 레포의 outputs/lora_adapter에서 확인하세요.')
print(f'학습 로그: {REPO / "fsd_training.log"}')

=== 최종 산출물 ===

✅ 파일이 있으면 이 레포의 outputs/lora_adapter에서 확인하세요.
학습 로그: /content/spatial-multimodal-2mo/fsd_training.log


In [17]:
# 6) 학습 로그와 최종 산출물을 Google Drive에 보존
from pathlib import Path
import shutil

DRIVE_BACKUP = Path('/content/drive/MyDrive/spatial-multimodal-2mo-results')
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)

log_source = REPO / 'fsd_training.log'
if log_source.exists():
    shutil.copy2(log_source, DRIVE_BACKUP / 'fsd_training.log')
    print(f'✅ 로그 복사: {DRIVE_BACKUP / "fsd_training.log"}')
else:
    print(f'⚠️ 로그 없음: {log_source}')

if OUTPUT_DIR.exists():
    adapter_backup = DRIVE_BACKUP / 'lora_adapter'
    shutil.copytree(OUTPUT_DIR, adapter_backup, dirs_exist_ok=True)
    files = sorted(path for path in adapter_backup.rglob('*') if path.is_file())
    print(f'✅ 산출물 복사: {adapter_backup} ({len(files)}개 파일)')
else:
    print(f'⚠️ 산출물 폴더 없음: {OUTPUT_DIR}')

print(f'📁 Drive 백업 위치: {DRIVE_BACKUP}')

✅ 로그 복사: /content/drive/MyDrive/spatial-multimodal-2mo-results/fsd_training.log
⚠️ 산출물 폴더 없음: /content/spatial-multimodal-2mo/outputs/lora_adapter
📁 Drive 백업 위치: /content/drive/MyDrive/spatial-multimodal-2mo-results
